In [1]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import pandas as pd
import numpy as np

from Preproces_prod2 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data,analyze_vrs_data,match_nn_max_dist_weigths,comparar_medias_test, charly_mip, charly_double_mip, mylogit, results_match, tabla_marcel, tabla_final,summary_eff,tabla_1_match

import inv

from IPython.core.display import display
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

np.random.seed(42)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

# df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_s39_2012.csv', index_col=0)
# df_upc_match_case = pd.read_csv(path_data/'df_upc_match_case_s39_2012.csv', index_col=0)

#df_vrs_match_case, df_upc_match_case = call_data('NAC_RNI_EGRESOS_ENTREGA_ISCI_11_04_2025_encr.csv')

df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_11_04_25.csv', index_col=0) 
df_upc_match_case = pd.read_csv(path_data/'df_upc_match_case_11_04_25.csv', index_col=0)

icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
    ################### AÑADIDOS POR MI #######################
    'A099', 
    'A080', 
    'A083', 
    'A082', 
    'A084',
    'A090', 
    'A081', 
    'A085'
]

ruts_cariopaticos_1 = pd.read_csv(path_data/'ruts_cardiopatias.csv', index_col=0).query('card1')
ruts_cariopaticos_2 = pd.read_csv(path_data/'ruts_cardiopatias.csv', index_col=0).query('card2')

list_experiments=[]


In [148]:
df = df_vrs_match_case.copy()

In [149]:
df_case = (df
           .query('fechaIng_vrs.notna()')
           .query("diag_vrs==True")
           .copy()
)

df_control = (df
              .query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))')
              .query("diag_vrs==False")
              .copy()
)

print(df_case.RUN.nunique(), df_control.RUN.nunique())
# VARIABLES MATCHING
distance_vars = ['edad_relativa','ingreso_relativo'] 
exact_var_match = ['region','group','prematuro']

# VARIABLES CONDITIONAL LOGIT
variables_logit = ['sexo','SEMANAS','PESO','cardio1']

matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                                                                               df_control=df_control, 
                                                                                                               distance_vars=distance_vars, 
                                                                                                               exact_var_match = exact_var_match, 
                                                                                                               ratio="1:1")

display(comparar_medias_test(df1=matched_data[~matched_data.Matched_Case_RUN.isna()],
                             df2=matched_data[matched_data.Matched_Case_RUN.isna()],
                             columnas=distance_vars + variables_logit))

df_matched_prterms = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched_prterms.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched_prterms, cases, controls,n_control_matched, n_case_matched, ratio,covs=variables_logit)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

1212 1642
creacion conjuntos A_i time: 2.8640270233154297
creacion variables time: 154.71951580047607
4.429033402539037
optimize model 1 time: 0.7949638366699219
optimize model 2 time: 2.572343349456787
matched_data time: 0.9128327369689941
Total cases = 1212, Total controls = 1642
Total cases matched is : 1112, Total control matched is : 1112
ratio: 1:1


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,edad_relativa,135.85,138.37,-0.66,0.51,
1,ingreso_relativo,71.55,109.15,-13.95,0.00,Existe una diferencia significativa.
2,sexo,0.60,0.62,-0.95,0.34,
3,SEMANAS,37.83,37.71,1.46,0.14,
4,PESO,3248.05,3188.59,2.43,0.02,Existe una diferencia significativa.
5,cardio1,0.01,0.03,-3.97,0.00,Existe una diferencia significativa.




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.840456
1    0.436199
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                        56  1056  1112
1                       295   817  1112
All                     351  1873  2224

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        15.954416   56.380139   50.0
1                        84.045584   43.619861   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.15
Efectividad:  85.3 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                 2224
Model:               ConditionalLogit   No. groups:                       1112
Log-Likelihood:             

In [150]:
tabla1 = tabla_1_match(df_matched_prterms)
display(tabla1)

Caso (n = 1112)  \
Variable                             Value                                                 
Age (days)                                                               140.00 (134.25)   
Sex                                                                                  NaN   
                                     Male                                    685 (61.6%)   
                                     Female                                  427 (38.4%)   
Gestational age at birth (weeks)                                            37.71 (2.29)   
Birth weight (grams)                                                    3188.59 (628.27)   
Risk groups                                                                          NaN   
                                     Congenital heart disease                  35 (3.1%)   
                                     Prematurity                             171 (15.4%)   
Macro-zone                                                                           NaN   
                                     Central Macrozone                       754 (67.8%)   
                                     South Macrozone                         306 (27.5%)   
                                     North Macrozone                           40 (3.6%)   
                                     Austral Macrozone                         12 (1.1%)   
PICU                                                                         230 (20.7%)   
Reason for hospital visit (controls)                                                 NaN   
                                     Other neonatal jaundice                           -   
                                     Urinary tract infection                           -   
                                     Other gastroenteritis and colitis                 -   
                                     Infectious diarrhoea                              -   
                                     Nausea and vomiting                               -   
                                     Other general symptoms and signs                  -   
                                     Other head injuries                               -   
                                     Abdominal pain                                    -   
                                     Feeding difficulties                              -   
                                     Polydipsia                                        -   

                                                                       Control (n = 1112)  
Variable                             Value                                                 
Age (days)                                                                131.50 (163.25)  
Sex                                                                                   NaN  
                                     Male                                     663 (59.6%)  
                                     Female                                   449 (40.4%)  
Gestational age at birth (weeks)                                             37.83 (1.66)  
Birth weight (grams)                                                     3248.05 (520.44)  
Risk groups                                                                           NaN  
                                     Congenital heart disease                    9 (0.8%)  
                                     Prematurity                              171 (15.4%)  
Macro-zone                                                                            NaN  
                                     Central Macrozone                        754 (67.8%)  
                                     South Macrozone                          306 (27.5%)  
                                     North Macrozone                            40 (3.6%)  
                                     Austral Macrozone                          12 (1.1%)  
PICU                    

In [151]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    tabla1.to_excel(writer, sheet_name="tabla_1")

# Intewnto multi table

In [4]:
def _tabla_1_single(
    df,
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    df = df.copy()
    df['grupo_tabla'] = np.where(df[id_col] == df[group_col], case_label, control_label)
    df['sexo'] = df['sexo'].map({0: 'Female', 1: 'Male'})

    variables_numericas_mediana = ['edad_relativa']
    variables_numericas_media = ['SEMANAS', 'PESO']
    variables_categoricas = ['sexo']
    macrozona_var = 'Macrozones'

    etiquetas_variables = {
        'edad_relativa': 'Age (days)',
        'SEMANAS': 'Gestational age at birth (weeks)',
        'PESO': 'Birth weight (grams)',
        'sexo': 'Sex',
        'Macrozones': 'Macro-zone',
        'cardio1': 'Congenital heart disease',
        'prematuro': 'Prematurity',
        'event_upc': 'PICU admission'
    }

    grupos = df['grupo_tabla'].unique()
    tabla = []
    index = []
    n_por_grupo = {g: len(df[df['grupo_tabla'] == g]) for g in grupos}
    nombres_columnas = {g: f"{g} (n = {n_por_grupo[g]})" for g in grupos}

    # ---- Variables numéricas con mediana (IQR)
    for var in variables_numericas_mediana:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            mediana = subset.median()
            iqr = subset.quantile(0.75) - subset.quantile(0.25)
            fila[nombres_columnas[g]] = f'{mediana:.{decimales}f} ({iqr:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Variables categóricas comunes
    for var in variables_categoricas:
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append({})  # Fila vacía para separar
        niveles = df[var].dropna().unique()
        for val in niveles:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset[var] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    # ---- Variables numéricas con media (SD)
    for var in variables_numericas_media:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            media = subset.mean()
            sd = subset.std()
            fila[nombres_columnas[g]] = f'{media:.{decimales}f} ({sd:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Risk groups (cardio1 y prematuro)
    index.append(('Risk groups', ''))
    tabla.append({})
    for var in ['cardio1', 'prematuro']:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[var] == 1).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append(('Risk groups', etiquetas_variables.get(var, var)))
        tabla.append(fila)

    # ---- Macrozonas
    index.append((etiquetas_variables.get(macrozona_var, macrozona_var), ''))
    tabla.append({})
    niveles = df[macrozona_var].dropna().unique()
    for val in niveles:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[macrozona_var] == val).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append((etiquetas_variables.get(macrozona_var, macrozona_var), val))
        tabla.append(fila)

    # ---- PICU
    fila = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g]
        total = len(subset)
        cuenta = (subset['cama'] == 'UPC').sum()
        porcentaje = 100 * cuenta / total if total > 0 else 0
        fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    index.append((etiquetas_variables.get('cama', 'PICU'), ''))
    tabla.append(fila)

    # ---- Diagnósticos - 2 grupos: diag_labels y diag_agrupao
    diag_labels = {
        'A090': 'Infectious diarrhoea',
        'R681': 'Other general symptoms and signs',
        'R11X': 'Nausea and vomiting',
        'R104': 'Abdominal pain',
        'S099': 'Other head injuries',
        'R634': 'Feeding difficulties',
        'R633': 'Polydipsia',
        'N390': 'Urinary tract infection',
        'P599': 'Other neonatal jaundice',
    }

    diag_agrupao = {
        'A099': 'Other gastroenteritis and colitis',
        'A080': 'Other gastroenteritis and colitis',
        'A083': 'Other gastroenteritis and colitis',
        'A082': 'Other gastroenteritis and colitis',
        'A084': 'Other gastroenteritis and colitis',
        'A085': 'Other gastroenteritis and colitis',
        'A081': 'Other gastroenteritis and colitis',
    }

    diag_frecuencias = []
    for diag_code, label in diag_labels.items():
        fila = {}
        count_control = 0
        for g in grupos:
            if g == case_label:
                fila[nombres_columnas[g]] = '-'  # Casos no exhiben "motivo"
            else:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control = cuenta
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        diag_frecuencias.append((count_control, ('Reason for hospital visit (controls)', label), fila))

    # Agrupación multiple: todos esos A09x a 'Other gastroenteritis and colitis'
    # calculamos la suma total para los 'controls'
    count_control_agrupao = 0
    for diag_code, label in diag_agrupao.items():
        for g in grupos:
            if g == control_label:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                # Reutilizamos 'total' con su última definición, pero OJO si hay varios grupos
                # asumiendo un solo 'control' group. Si tienes 2+ grupos de control habría que ampliarlo
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control_agrupao += cuenta

    fila_agrupao = {}
    porcentaje_agrupao = 100 * count_control_agrupao / total if total > 0 else 0
    # Asignamos al control
    fila_agrupao[nombres_columnas[control_label]] = f'{count_control_agrupao} ({porcentaje_agrupao:.1f}%)'
    # Casos sin "motivo"
    fila_agrupao[nombres_columnas[case_label]] = '-'
    # Agregamos a diag_frecuencias
    diag_frecuencias.append((
        count_control_agrupao,
        ('Reason for hospital visit (controls)', 'Other gastroenteritis and colitis'),
        fila_agrupao
    ))

    # Insertamos encabezado en la tabla
    index.append(('Reason for hospital visit (controls)', ''))
    tabla.append({})

    # Ordenamos diag_frecuencias por conteo y actualizamos la tabla
    for _, idx_, fila_ in sorted(diag_frecuencias, key=lambda x: x[0], reverse=True):
        index.append(idx_)
        tabla.append(fila_)

    multiindex = pd.MultiIndex.from_tuples(index, names=['Variable', 'Value'])
    return pd.DataFrame(tabla, index=multiindex)


In [2]:
def _tabla_1_single(
    df,
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    import numpy as np
    import pandas as pd

    df = df.copy()
    df['grupo_tabla'] = np.where(df[id_col] == df[group_col], case_label, control_label)
    df['sexo'] = df['sexo'].map({0: 'Female', 1: 'Male'})

    variables_numericas_mediana = ['edad_relativa']
    variables_numericas_media = ['SEMANAS', 'PESO']
    variables_categoricas = ['sexo']
    macrozona_var = 'Macrozones'

    etiquetas_variables = {
        'edad_relativa': 'Age (days)',
        'SEMANAS': 'Gestational age at birth (weeks)',
        'PESO': 'Birth weight (grams)',
        'sexo': 'Sex',
        'Macrozones': 'Macro-zone',
        'cardio1': 'Congenital heart disease',
        'prematuro': 'Prematurity',
        'event_upc': 'PICU admission'
    }

    grupos = df['grupo_tabla'].unique()
    tabla = []
    index = []
    n_por_grupo = {g: len(df[df['grupo_tabla'] == g]) for g in grupos}
    nombres_columnas = {g: f"{g} (n = {n_por_grupo[g]})" for g in grupos}

    # ---- Variables numéricas con mediana (IQR)
    for var in variables_numericas_mediana:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            mediana = subset.median()
            iqr = subset.quantile(0.75) - subset.quantile(0.25)
            fila[nombres_columnas[g]] = f'{mediana:.{decimales}f} ({iqr:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Variables categóricas comunes
    for var in variables_categoricas:
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append({})  # Fila vacía para separar
        niveles = df[var].dropna().unique()
        for val in niveles:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset[var] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    # ---- Variables numéricas con media (SD)
    for var in variables_numericas_media:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            media = subset.mean()
            sd = subset.std()
            fila[nombres_columnas[g]] = f'{media:.{decimales}f} ({sd:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Risk groups (cardio1 y prematuro)
    index.append(('Risk groups', ''))
    tabla.append({})
    for var in ['cardio1', 'prematuro']:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[var] == 1).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append(('Risk groups', etiquetas_variables.get(var, var)))
        tabla.append(fila)

    # ---- Macrozonas
    index.append((etiquetas_variables.get(macrozona_var, macrozona_var), ''))
    tabla.append({})
    niveles = df[macrozona_var].dropna().unique()
    for val in niveles:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[macrozona_var] == val).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append((etiquetas_variables.get(macrozona_var, macrozona_var), val))
        tabla.append(fila)

    # ---- PICU
    fila = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g]
        total = len(subset)
        cuenta = (subset['cama'] == 'UPC').sum()
        porcentaje = 100 * cuenta / total if total > 0 else 0
        fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    index.append((etiquetas_variables.get('cama', 'PICU'), ''))
    tabla.append(fila)

    # -------------------------------------------------------------------------
    # A) Nirsevimab Recipients (inmunizado=1 => "Yes", inmunizado=0 => "No")
    # -------------------------------------------------------------------------
    # Agregamos un bloque para la variable 'inmunizado'
    index.append(("Nirsevimab Recipients", ""))
    tabla.append({})  # Fila vacía para separar visualmente

    # 1. Bloque categórico "Yes / No"
    for val_inm, label_inm in [(1, "Yes"), (0, "No")]:
        fila_inm = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            if 'inmunizado' not in subset.columns:
                # Si la columna no existe, forzamos 0
                fila_inm[nombres_columnas[g]] = "- (No data)"
            else:
                cuenta = (subset['inmunizado'] == val_inm).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila_inm[nombres_columnas[g]] = f"{cuenta} ({porcentaje:.1f}%)"
        index.append(("Nirsevimab Recipients", label_inm))
        tabla.append(fila_inm)

    # 2. Bloque "Time immunized (days)"
    #    Resta 'fechaIng_any' - 'fecha_Inm'. Si no hay datos, "-", sino media en días
    fila_time = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g].copy()

       # if "fecha_Inm" not in subset.columns or "fechaIng_any" not in subset.columns:
            # No existen las columnas => "-"
            #fila_time[nombres_columnas[g]] = "-"
            #continue

        # Convertir a días la diferencia
        # Solo para filas donde existan ambas fechas
        #def compute_days(row):
            #if pd.isna(row.get("fecha_Inm")) or pd.isna(row.get("fechaIng_any")):
                #return np.nan
            # asumiendo que son datetime, la resta da Timedelta
            #return (row["fechaIng_any"] - row["fecha_Inm"]).days

       # subset["time_inm"] = subset.apply(compute_days, axis=1)
        # Filtramos NA para la media
        valid_times = subset["time_inm"].dropna()
        if valid_times.empty:
            fila_time[nombres_columnas[g]] = "-"
        else:
            mean_days = valid_times.mean()
            fila_time[nombres_columnas[g]] = f"{mean_days:.{decimales}f}"
    index.append(("Nirsevimab Recipients", "Days from immunization to admission"))
    tabla.append(fila_time)

    # -------------------------------------------------------------------------
    # ---- Diagnósticos - 2 grupos: diag_labels y diag_agrupao
    # -------------------------------------------------------------------------
    diag_labels = {
        'A090': 'Infectious diarrhoea',
        'R681': 'Other general symptoms and signs',
        'R11X': 'Nausea and vomiting',
        'R104': 'Abdominal pain',
        'S099': 'Other head injuries',
        'R634': 'Feeding difficulties',
        'R633': 'Polydipsia',
        'N390': 'Urinary tract infection',
        'P599': 'Other neonatal jaundice',
    }

    diag_agrupao = {
        'A099': 'Other gastroenteritis and colitis',
        'A080': 'Other gastroenteritis and colitis',
        'A083': 'Other gastroenteritis and colitis',
        'A082': 'Other gastroenteritis and colitis',
        'A084': 'Other gastroenteritis and colitis',
        'A085': 'Other gastroenteritis and colitis',
        'A081': 'Other gastroenteritis and colitis',
    }

    diag_frecuencias = []
    for diag_code, label in diag_labels.items():
        fila_diag = {}
        count_control = 0
        for g in grupos:
            if g == case_label:
                fila_diag[nombres_columnas[g]] = '-'  # Casos no exhiben "motivo"
            else:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control = cuenta
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila_diag[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        diag_frecuencias.append((count_control, ('Reason for hospital visit (controls)', label), fila_diag))

    # Agrupación multiple: todos esos A09x a 'Other gastroenteritis and colitis'
    # calculamos la suma total para los 'controls'
    count_control_agrupao = 0
    for diag_code, label in diag_agrupao.items():
        for g in grupos:
            if g == control_label:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control_agrupao += cuenta

    fila_agrupao = {}
    porcentaje_agrupao = 100 * count_control_agrupao / total if total > 0 else 0
    # Asignamos al control
    fila_agrupao[nombres_columnas[control_label]] = f'{count_control_agrupao} ({porcentaje_agrupao:.1f}%)'
    # Casos sin "motivo"
    fila_agrupao[nombres_columnas[case_label]] = '-'
    diag_frecuencias.append((
        count_control_agrupao,
        ('Reason for hospital visit (controls)', 'Other gastroenteritis and colitis'),
        fila_agrupao
    ))

    # Insertamos encabezado en la tabla
    index.append(('Reason for hospital visit (controls)', ''))
    tabla.append({})

    # Ordenar por conteo y actualizar la tabla
    for _, idx_, fila_ in sorted(diag_frecuencias, key=lambda x: x[0], reverse=True):
        index.append(idx_)
        tabla.append(fila_)

    multiindex = pd.MultiIndex.from_tuples(index, names=['Variable', 'Value'])
    return pd.DataFrame(tabla, index=multiindex)


In [11]:
def _tabla_1_single(
    df,
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    import numpy as np
    import pandas as pd

    df = df.copy()
    df['grupo_tabla'] = np.where(df[id_col] == df[group_col], case_label, control_label)
    df['sexo'] = df['sexo'].map({0: 'Female', 1: 'Male'})

    variables_numericas_mediana = ['edad_relativa']
    variables_numericas_media = ['SEMANAS', 'PESO']
    variables_categoricas = ['sexo']
    macrozona_var = 'Macrozones'

    etiquetas_variables = {
        'edad_relativa': 'Age (days)',
        'SEMANAS': 'Gestational age at birth (weeks)',
        'PESO': 'Birth weight (grams)',
        'sexo': 'Sex',
        'Macrozones': 'Macro-zone',
        'cardio1': 'Congenital heart disease',
        'prematuro': 'Prematurity',
        'event_upc': 'PICU admission'
    }

    grupos = df['grupo_tabla'].unique()
    tabla = []
    index = []
    n_por_grupo = {g: len(df[df['grupo_tabla'] == g]) for g in grupos}
    nombres_columnas = {g: f"{g} (n = {n_por_grupo[g]})" for g in grupos}

    # ---- Variables numéricas con mediana (IQR)
    for var in variables_numericas_mediana:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            mediana = subset.median()
            iqr = subset.quantile(0.75) - subset.quantile(0.25)
            fila[nombres_columnas[g]] = f'{mediana:.{decimales}f} ({iqr:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Variables categóricas comunes
    for var in variables_categoricas:
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append({})  # Fila vacía para separar
        niveles = df[var].dropna().unique()
        for val in niveles:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset[var] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    # ---- Variables numéricas con media (SD)
    for var in variables_numericas_media:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            media = subset.mean()
            sd = subset.std()
            fila[nombres_columnas[g]] = f'{media:.{decimales}f} ({sd:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    # ---- Risk groups (cardio1 y prematuro)
    index.append(('Risk groups', ''))
    tabla.append({})
    for var in ['cardio1', 'prematuro']:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[var] == 1).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append(('Risk groups', etiquetas_variables.get(var, var)))
        tabla.append(fila)

    # ---- Macrozonas
    index.append((etiquetas_variables.get(macrozona_var, macrozona_var), ''))
    tabla.append({})
    niveles = df[macrozona_var].dropna().unique()
    for val in niveles:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[macrozona_var] == val).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append((etiquetas_variables.get(macrozona_var, macrozona_var), val))
        tabla.append(fila)

    # ---- PICU
    fila = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g]
        total = len(subset)
        cuenta = (subset['cama'] == 'UPC').sum()
        porcentaje = 100 * cuenta / total if total > 0 else 0
        fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    index.append((etiquetas_variables.get('cama', 'PICU'), ''))
    tabla.append(fila)

    # -------------------------------------------------------------------------
    # A) Nirsevimab Recipients (inmunizado=1 => "Yes", inmunizado=0 => "No")
    # -------------------------------------------------------------------------
    # Agregamos un bloque para la variable 'inmunizado'
    index.append(("Nirsevimab Recipients", ""))
    tabla.append({})  # Fila vacía para separar visualmente

    # 1. Bloque categórico "Yes / No"
    for val_inm, label_inm in [(1, "Yes"), (0, "No")]:
        fila_inm = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            if 'inmunizado' not in subset.columns:
                # Si la columna no existe, forzamos 0
                fila_inm[nombres_columnas[g]] = "- (No data)"
            else:
                cuenta = (subset['inmunizado'] == val_inm).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila_inm[nombres_columnas[g]] = f"{cuenta} ({porcentaje:.1f}%)"
        index.append(("Nirsevimab Recipients", label_inm))
        tabla.append(fila_inm)

    # 2. Bloque "Time immunized (days)"
    #    Resta 'fechaIng_any' - 'fecha_Inm'. Si no hay datos, "-", sino media en días

    # -------------------------------------------------------------------------
    # ---- Diagnósticos - 2 grupos: diag_labels y diag_agrupao
    # -------------------------------------------------------------------------
    diag_labels = {
        'A090': 'Infectious diarrhoea',
        'R681': 'Other general symptoms and signs',
        'R11X': 'Nausea and vomiting',
        'R104': 'Abdominal pain',
        'S099': 'Other head injuries',
        'R634': 'Feeding difficulties',
        'R633': 'Polydipsia',
        'N390': 'Urinary tract infection',
        'P599': 'Other neonatal jaundice',
    }

    diag_agrupao = {
        'A099': 'Other gastroenteritis and colitis',
        'A080': 'Other gastroenteritis and colitis',
        'A083': 'Other gastroenteritis and colitis',
        'A082': 'Other gastroenteritis and colitis',
        'A084': 'Other gastroenteritis and colitis',
        'A085': 'Other gastroenteritis and colitis',
        'A081': 'Other gastroenteritis and colitis',
    }

    diag_frecuencias = []
    for diag_code, label in diag_labels.items():
        fila_diag = {}
        count_control = 0
        for g in grupos:
            if g == case_label:
                fila_diag[nombres_columnas[g]] = '-'  # Casos no exhiben "motivo"
            else:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control = cuenta
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila_diag[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        diag_frecuencias.append((count_control, ('Reason for hospital visit (controls)', label), fila_diag))

    # Agrupación multiple: todos esos A09x a 'Other gastroenteritis and colitis'
    # calculamos la suma total para los 'controls'
    count_control_agrupao = 0
    for diag_code, label in diag_agrupao.items():
        for g in grupos:
            if g == control_label:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control_agrupao += cuenta

    fila_agrupao = {}
    porcentaje_agrupao = 100 * count_control_agrupao / total if total > 0 else 0
    # Asignamos al control
    fila_agrupao[nombres_columnas[control_label]] = f'{count_control_agrupao} ({porcentaje_agrupao:.1f}%)'
    # Casos sin "motivo"
    fila_agrupao[nombres_columnas[case_label]] = '-'
    diag_frecuencias.append((
        count_control_agrupao,
        ('Reason for hospital visit (controls)', 'Other gastroenteritis and colitis'),
        fila_agrupao
    ))

    # Insertamos encabezado en la tabla
    index.append(('Reason for hospital visit (controls)', ''))
    tabla.append({})

    # Ordenar por conteo y actualizar la tabla
    for _, idx_, fila_ in sorted(diag_frecuencias, key=lambda x: x[0], reverse=True):
        index.append(idx_)
        tabla.append(fila_)

    multiindex = pd.MultiIndex.from_tuples(index, names=['Variable', 'Value'])
    return pd.DataFrame(tabla, index=multiindex)

In [12]:
def tabla_1_match_multi(
    df1=None, label1="base1",
    df2=None, label2="base2",
    df3=None, label3="base3",
    df4=None, label4="base4",
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
):
    """
    Genera la tabla 1 para hasta 4 bases de datos. Si se pasa sólo df1, 
    reproduce la tabla de la función original. Para 2, 3, o 4, concatena 
    horizontalmente las tablas.
    """
    # Lista de (df, label)
    df_list = []
    if df1 is not None:
        df_list.append((df1, label1))
    if df2 is not None:
        df_list.append((df2, label2))
    if df3 is not None:
        df_list.append((df3, label3))
    if df4 is not None:
        df_list.append((df4, label4))

    if not df_list:
        # No se pasó ninguna base, retornar None
        print("No se recibió ninguna base de datos.")
        return None

    # Procesar cada base y renombrar columnas
    tables = []
    for (dfx, labelx) in df_list:
        single_table = _tabla_1_single(
            dfx,
            id_col=id_col,
            group_col=group_col,
            case_label=case_label,
            control_label=control_label,
            decimales=decimales
        )
        # Renombrar columnas con un MultiIndex: (label, col)
        # Esto evita colisiones si varias bases tienen las mismas columnas
        single_table.columns = pd.MultiIndex.from_product([[labelx], single_table.columns])
        tables.append(single_table)

    if len(tables) == 1:
        # Si sólo hay 1 tabla, la devolvemos tal cual
        return tables[0]
    else:
        # Concatenar horizontalmente
        df_final = pd.concat(tables, axis=1)
        return df_final


In [13]:
df1 = pd.read_csv(path_data/'matched_data_VRS_general_francia.csv',index_col=0).assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days)
df2 = pd.read_csv(path_data/'matched_data_prematuros_all_born.csv',index_col=0).assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days)
df3 = pd.read_csv(path_data/'matched_data_cardiopats_1_all_born.csv',index_col=0).assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days)

Tabla_1 = tabla_1_match_multi(
    df1=df1, label1="All",
    df2=df2, label2="preterms",
    df3=df3, label3="Heart-congenital-disease",
    df4=None, label4="base4",
    id_col='RUN',
    group_col='Group',
    case_label='Caso',
    control_label='Control',
    decimales=2
)

In [14]:
Tabla_1

All  \
                                                                         Caso (n = 1112)   
Variable                             Value                                                 
Age (days)                                                               140.00 (134.25)   
Sex                                                                                  NaN   
                                     Male                                    685 (61.6%)   
                                     Female                                  427 (38.4%)   
Gestational age at birth (weeks)                                            37.71 (2.29)   
Birth weight (grams)                                                    3188.59 (628.27)   
Risk groups                                                                          NaN   
                                     Congenital heart disease                  35 (3.1%)   
                                     Prematurity                             171 (15.4%)   
Macro-zone                                                                           NaN   
                                     Central Macrozone                       754 (67.8%)   
                                     South Macrozone                         306 (27.5%)   
                                     North Macrozone                           40 (3.6%)   
                                     Austral Macrozone                         12 (1.1%)   
PICU                                                                         230 (20.7%)   
Nirsevimab Recipients                                                                NaN   
                                     Yes                                     817 (73.5%)   
                                     No                                      295 (26.5%)   
Reason for hospital visit (controls)                                                 NaN   
                                     Other neonatal jaundice                           -   
                                     Urinary tract infection                           -   
                                     Other gastroenteritis and colitis                 -   
                                     Infectious diarrhoea                              -   
                                     Nausea and vomiting                               -   
                                     Other general symptoms and signs                  -   
                                     Other head injuries                               -   
                                     Abdominal pain                                    -   
                                     Feeding difficulties                              -   
                                     Polydipsia                                        -   

                                                                                           \
                                                                       Control (n = 1112)   
Variable                             Value                                                  
Age (days)                                                                131.50 (163.25)   
Sex                                                                                   NaN   
                                     Male                                     663 (59.6%)   
                                     Female                                   449 (40.4%)   
Gestational age at birth (weeks)                                             37.83 (1.66)   
Birth weight (grams)                                                     3248.05 (520.44)   
Risk groups                                                                           NaN   
                                     Congenital heart disease                    9 (0.8%)   
                                     Prematurity                              171 (15.4%)   
Macro-zone              

In [15]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Tabla_1.to_excel(writer, sheet_name="Multi_tabla_1_v2")

In [48]:
df1 = pd.read_csv(path_data/'matched_data_VRS_general_francia.csv',index_col=0).assign(time_inm = lambda x: (pd.to_datetime(x['fechaIng_any']) - pd.to_datetime(x['fechaInm'])).dt.days)

In [54]:
df1.assign(birth_to_inmune = lambda x: (pd.to_datetime(x['fechaInm']) - pd.to_datetime(x['fecha_nac'])).dt.days,
           grupito = lambda x: x.Matched_Case_RUN.notna())[['birth_to_inmune','time_inm','ingreso_relativo','grupito']].groupby('grupito').mean()

,birth_to_inmune,time_inm,ingreso_relativo
grupito,,,
False,44.488372,72.538556,109.147482
True,67.750000,21.841856,71.553957


In [52]:
df1

,RUN,RUN_RNI,RUN_M,VACUNADO,MARCA,FECHA_NACIMIENTO,ANO_NAC,SEXO,SEMANAS,PESO,TALLA,EDAD_M,INS_C_M,INS_N_M,COMUNA,COMUNA_N,REG_RES,URBA_RURAL,NAC_MA,FECHA_INMUNIZACION,FECHA_DEFUNCION,CAUSA_DEFUNCION,VIVO,FALLECIDO_PREVIO,ESTAB,ServicioSalud,Seremi,P_ORIGEN,PREVI,FECHA_INGRESO,FECHA_EGRESO,AREA_FUNC_I,SER_CLIN_I,DIAS_ESTAD,COND_EGR,DIAG1,DIAG2,DIAG3,DIAG4,DIAG5,DIAG6,DIAG7,DIAG8,DIAG9,DIAG10,DIAG11,NOMBRE_REGION,Porcentaje Urbano,porcent_rural,percent_poor_multidim,percent_poor,p_00001_lognormal,p_99999_lognormal,fecha_nac,fechaIng_any,fechaEgr,fechaInm,VRS_D1,VRS_D1y3,diag_irag,diag_ira_alta,LRTI_Flag,LRTI_all_j,fechaIng_vrs,fechaIng_LRTI,group,sexo,prematuro_extremo,muy_prematuro,prematuro_moderado,prematuro,atypic_mom_age,region,Macrozona2,cama,fecha_upc,fecha_upc_vrs,days_upc,days_estad_vrs,is_rural,categori_macro,categori_regions,exp_rural,is_poor,vrs_pre_campaña,lrti_pre_campaña,any_pre_campaña,upc_pre_campaña,days_upc_vrs,event_upc,event,event_LRTI,event_any,take_nirse,cardio1,log_peso,Macrozones,fechaIng_vrs_copy,inmunizado,chile_chico,estado_inmunizacion,diag_vrs,diag_1_leter,edad_relativa,ingreso_relativo,Matched_Case_RUN,Group,time_inm
0,3f87a6a80e3e29a2a98e8e4e0118b9971afcd6092f48b2378ba20df75fa2400c,1.0,75b718e11e4b439f1b27eddff30a1535db823a309844cdc213960b5779708ac1,SI,0.0,12Mar2024,2024.0,1.0,38.0,3550.0,49.0,26.0,3.0,1.0,13401.0,13401.0,13.0,0.0,C,2024-05-17,NaN,NaN,SI,VIVO,113180.0,13.0,NaN,152.0,1.0,01Apr2024,04Apr2024,0,NaN,3.0,1.0,J210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Región Metropolitana de Santiago,0.986754,0.013246,0.187083,5.986493,1873.189390,5271.639121,2024-03-12,2024-04-01,2024-04-04,NaN,1,1,True,False,True,True,2024-04-01,2024-04-01,CATCH_UP,1,0,0,0,0,0,METROPOLITANA,Centro,NaN,NaN,NaN,0,3.0,0,1,12,1.013334,1,0,0,0,0,0,0,1,1,1,True,0,8.174703,Central Macrozone,2024-04-01,0,1,0,1,J,136,0.0,NaN,3f87a6a80e3e29a2a98e8e4e0118b9971afcd6092f48b2378ba20df75fa2400c,NaN
1,9e8f2e295b2370fb262c23d23627fa8c509192d6aae53d2c8dfdfc3c39dfb79b,1.0,76e10b855c76d53866ea816550181fb3e48622390be18947625ed923f5a74045,SI,0.0,19Mar2024,2024.0,2.0,39.0,3660.0,50.0,34.0,4.0,2.0,13401.0,13401.0,13.0,0.0,C,2024-05-24,NaN,NaN,SI,VIVO,113180.0,13.0,NaN,152.0,1.0,01Apr2024,02Apr2024,0,NaN,1.0,1.0,J210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Región Metropolitana de Santiago,0.986754,0.013246,0.187083,5.986493,1994.452036,5640.163672,2024-03-19,2024-04-01,2024-04-02,NaN,1,1,True,False,True,True,2024-04-01,2024-04-01,CATCH_UP,0,0,0,0,0,0,METROPOLITANA,Centro,NaN,NaN,NaN,0,1.0,0,1,12,1.013334,1,0,0,0,0,0,0,1,1,1,True,0,8.205218,Central Macrozone,2024-04-01,0,1,0,1,J,143,0.0,NaN,9e8f2e295b2370fb262c23d23627fa8c509192d6aae53d2c8dfdfc3c39dfb79b,NaN
2,1a27a4c3503b760fb11ca154a007ba30f208f877c5cdafd2c48b8315ab5c9edc,0.0,66458cce4ab8f603af62bb9432302324e6f08a3661562c1a65c0705aa42c9cc7,NO,0.0,11Mar2024,2024.0,1.0,37.0,2950.0,49.0,37.0,4.0,2.0,8103.0,8103.0,8.0,0.0,C,NaN,NaN,NaN,SI,VIVO,118100.0,18.0,NaN,152.0,1.0,01Apr2024,03Apr2024,0,NaN,2.0,1.0,J219,NaN,B348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Región Del Bíobío,0.997934,0.002066,0.108700,5.710677,1749.884992,4899.237174,2024-03-11,2024-04-01,2024-04-03,NaN,1,1,True,False,True,True,2024-04-01,2024-04-01,CATCH_UP,1,0,0,0,0,0,BIOBIO,Centro,NaN,NaN,NaN,0,2.0,0,1,6,1.002068,0,0,0,0,0,0,0,1,1,1,False,0,7.989560,South Macrozone,2024-04-01,0,0,0,1,J,135,0.0,NaN,1a27a4c3503b760fb11ca154a007ba30f208f877c5cdafd2c48b8315ab5c9edc,NaN
3,c9cc7a7421b5ca1dcc2a196ecf72d9287be02e5130dbf7fef1e24ff8d58ef765,1.0,60526ae339f710ea1b4d9cb0cf050df2496ec7fdc6a5370989e89c815b3b14e0,SI,0.0,17Feb2024,2024.0,1.0,38.0,3090.0,50.0,19.0,4.0,2.0,7201.0,7201.0,7.0,0.0,C,2024-07-11,NaN,NaN,SI,VIVO,116111.0,16.0,NaN,152.0,1.0,01Apr2024,11Apr2024,0,NaN,10.0,1.0,J219,NaN,U071,R160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Región Del Maule,0.825541,0.174459,0.161226,11.572210,1873.189390,5271.639121,2024-02-17,2024-04-01,2024-04-11,NaN,1,1,True,False,True,True,2024-04-01,2024-04-01,CATCH_UP,1,0,0,0,0,1,MAULE,Centro,NaN,NaN,NaN,0,10.0,0,1,11,